# Scrollie — Hirriririir Fat/Water Segmentation Viewer

- **Left**: original Fat or Water image
- **Right**: Hirriririir segmentation overlay

In [ ]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as _cm
import SimpleITK as sitk
from ipywidgets import IntSlider, Dropdown, VBox
import ipywidgets as widgets
from IPython.display import display

In [ ]:
MODALITY = 'WATER'   # 'FAT', 'WATER', or 'FATFRAC'

DATA_ROOT = 'myosegmenTUM'

if MODALITY == 'FATFRAC':
    SEG_DIR   = 'multimodal_thigh_segs_fatfrac_v2'
    MOD_TOKEN = 'FATFRACTION'
else:
    SEG_DIR   = f'multimodal_thigh_segs_{MODALITY.lower()}'
    MOD_TOKEN = MODALITY

LABEL_MAP = {
    1:  'Sartorius',
    2:  'Rectus_Femoris',
    3:  'Vastus_Lateralis',
    4:  'Vastus_Intermedius',
    5:  'Vastus_Medialis',
    6:  'Adductor_Magnus',
    7:  'Gracilis',
    8:  'Biceps_Femoris_Long',
    9:  'Semitendinosus',
    10: 'Semimembranosus',
    11: 'Biceps_Femoris_Short',
}

cmap = _cm.get_cmap('tab20', len(LABEL_MAP))

legend_patches = [
    mpatches.Patch(color=cmap(i), alpha=0.6, label=name)
    for i, name in enumerate(LABEL_MAP.values())
]

seg_files    = sorted(glob.glob(os.path.join(SEG_DIR, '*_thigh_seg.nii.gz')))
file_options = {
    os.path.basename(p).replace('_thigh_seg.nii.gz', ''): p
    for p in seg_files
}

def seg_stem_to_image(stem):
    stack_n = re.search(r'stack(\d+)', stem).group(1)
    subject = stem.split(f'_{MOD_TOKEN}')[0]
    return os.path.join(DATA_ROOT, subject, 'ImageData',
                        f'{subject}_{MOD_TOKEN}',
                        f'{subject}_{MOD_TOKEN}_stack{stack_n}.nii')

print(f'Modality : {MODALITY}')
print(f'Seg dir  : {os.path.abspath(SEG_DIR)}')
print(f'Found    : {len(file_options)} segmentation files')
if file_options:
    first = list(file_options)[0]
    print(f'Image path example: {seg_stem_to_image(first)}')

In [ ]:
def build_overlay(seg_array):
    overlay = np.zeros((*seg_array.shape, 4), dtype=float)
    for i, (label_idx, _) in enumerate(LABEL_MAP.items()):
        color = cmap(i)
        overlay[seg_array == label_idx] = [color[0], color[1], color[2], 0.5]
    return overlay

def load_stack(label):
    seg_path = file_options[label]
    nii_path = seg_stem_to_image(label)
    img_arr  = sitk.GetArrayFromImage(sitk.ReadImage(nii_path)).astype(float)
    img_norm = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-8)
    seg_arr  = sitk.GetArrayFromImage(sitk.ReadImage(seg_path))
    overlay  = build_overlay(seg_arr)
    return img_norm, overlay

In [ ]:
file_dropdown = Dropdown(options=list(file_options.keys()), description='Stack:')
slice_slider  = IntSlider(min=0, max=1, step=1, value=0, description='Slice:',
                          layout=widgets.Layout(width='600px'))
out = widgets.Output()

_cache = {}

def get_data(label):
    if label not in _cache:
        img_norm, overlay = load_stack(label)
        _cache[label] = (img_norm, overlay)
        slice_slider.max   = img_norm.shape[0] - 1
        slice_slider.value = 0
    return _cache[label]

def render(label, slice_idx):
    img_norm, overlay = get_data(label)
    img = img_norm[slice_idx]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(img, cmap='gray', origin='lower')
    axes[0].set_title(f'{MODALITY} image — slice {slice_idx}')
    axes[0].axis('off')

    axes[1].imshow(img, cmap='gray', origin='lower')
    axes[1].imshow(overlay[slice_idx], origin='lower')
    axes[1].set_title('Hirriririir segmentation')
    axes[1].axis('off')
    axes[1].legend(handles=legend_patches, loc='lower right', fontsize=6, framealpha=0.7)

    fig.suptitle(label, fontsize=10)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

def on_file_change(change):
    _cache.clear()
    get_data(change['new'])
    render(file_dropdown.value, slice_slider.value)

def on_slice_change(change):
    render(file_dropdown.value, change['new'])

file_dropdown.observe(on_file_change, names='value')
slice_slider.observe(on_slice_change, names='value')

if file_options:
    get_data(file_dropdown.value)
    render(file_dropdown.value, 0)

display(VBox([file_dropdown, slice_slider, out]))